# difmap-python — Notebook de démonstration

Workflow complet d'imagerie radio-interférométrique avec `DifmapSession`.

| Étape | Méthode |
|---|---|
| Chargement | `s.observe()` |
| Sélection polarisation | `s.obs.select()` |
| UV Coverage | `s.vis.uvplot()` |
| Amplitude vs rayon UV | `s.vis.radplot()` |
| Dirty Map | `s.imager.invert()` |
| Déconvolution CLEAN | `s.imager.clean()` |
| Auto-calibration | `s.imager.selfcal()` |
| Carte finale | `s.imager.restore()` |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'builddir'))

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

from difmap_wrapper.core.session import DifmapSession

DifmapSession._instance = None
print('Imports OK')

---
## Exemple minimal

La forme la plus simple — chargement, sélection, dirty map :

In [ ]:
with DifmapSession() as s:
    s.observe('tests/test_data/0003-066_X.SPLIT.1')
    img = s.imager.make_dirty_map(512, 0.1, pol='RR')

    plt.figure(figsize=(6, 5))
    plt.imshow(img['data'], origin='lower', cmap='afmhot', extent=img['extent'])
    plt.colorbar(label='Jy/beam')
    plt.title(f"Dirty Map — {s.obs.source}")
    plt.xlabel('RA offset (mas)')
    plt.ylabel('Dec offset (mas)')
    plt.tight_layout()
    plt.show()

---
## Workflow complet

Les cellules suivantes détaillent chaque étape séparément.

### 1 · Chargement

In [ ]:
DATA_FILE  = 'tests/test_data/0003-066_X.SPLIT.1'
MAP_SIZE   = 512
CELLSIZE   = 0.1   # mas/pixel

s = DifmapSession()
s.observe(DATA_FILE)

print(f'Source       : {s.obs.source}')
print(f'Sous-réseaux : {s.obs.nsub()}')

### 2 · Sélection de la polarisation

In [ ]:
pol = s.obs.select(pol='I', ifs=(1, 0), channels=(1, 0))

data    = s.obs.get_data()
n_vis   = len(data['u'])
uv_max  = np.sqrt(data['u']**2 + data['v']**2).max() / 1e6

print(f'Polarisation : {pol}')
print(f'Visibilités  : {n_vis:,}')
print(f'UV max       : {uv_max:.0f} Mλ')
print(f'Amp médiane  : {np.median(data["amp"]):.4f} Jy')

### 3 · UV Coverage

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
s.vis.uvplot(ax=ax, color='steelblue', s=1.5, alpha=0.45,
             title=f'UV Coverage — {s.obs.source} ({pol})')
plt.tight_layout()
plt.show()

### 4 · Radplot — amplitude vs rayon UV

Une source ponctuelle donne une ligne plate ; une structure étendue montre une décroissance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
s.vis.radplot(ax=ax, color='#222222', s=1, alpha=0.35,
              title=f'Amplitude vs Rayon UV — {s.obs.source} ({pol})')
plt.tight_layout()
plt.show()

### 5 · Dirty Map

Transformée de Fourier inverse sans déconvolution.

In [ ]:
s.imager.uvweight(bin_size=2.0, err_power=0.0)
s.imager.mapsize(MAP_SIZE, CELLSIZE)
s.imager.invert()

dirty   = s.imager.get_map_package(cellsize=CELLSIZE)
p_dirty = s.imager.peak()

print(f"Dirty Map — Pic : {p_dirty['flux']:.4f} Jy/beam")
print(f"            RMS : {p_dirty['rms']:.4f} Jy/beam")
print(f"            SNR : {p_dirty['snr']:.1f}")
print(f"Beam : {dirty['info']['bmaj']:.3f} × {dirty['info']['bmin']:.3f} mas  BPA={dirty['info']['bpa']:.1f}°")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dirty Beam — crop central
beam = dirty['beam_data']
bh, bw = beam.shape
cy_b, cx_b = bh // 2, bw // 2
crop = min(64, cy_b, cx_b)
beam_crop = beam[cy_b-crop:cy_b+crop, cx_b-crop:cx_b+crop]
ext_beam = [v * (2 * crop / bw) for v in dirty['extent']]
im_b = axes[0].imshow(beam_crop, origin='lower', cmap='RdBu_r',
                       extent=ext_beam, vmin=-0.3, vmax=1.0)
axes[0].set_title('Dirty Beam (PSF)')
axes[0].set_xlabel('RA offset (mas)')
axes[0].set_ylabel('Dec offset (mas)')
plt.colorbar(im_b, ax=axes[0], label='Amplitude normalisée')

# Dirty Map
vmax = np.percentile(dirty['data'], 99.5)
im_d = axes[1].imshow(dirty['data'], origin='lower', cmap='afmhot',
                       extent=dirty['extent'], vmin=0, vmax=vmax)
axes[1].set_title(f"Dirty Map — {s.obs.source}")
axes[1].set_xlabel('RA offset (mas)')
axes[1].set_ylabel('Dec offset (mas)')
plt.colorbar(im_d, ax=axes[1], label='Jy/beam')

plt.suptitle('Avant déconvolution CLEAN', fontsize=14)
plt.tight_layout()
plt.show()

### 6 · Boucle CLEAN + Auto-calibration

```
invert → peakwin → clean → selfcal(phase) × N → selfcal(amp+phase) × M → restore
```

In [ ]:
NITER        = 200
GAIN         = 0.05
WIN_SIZE     = 2.0
N_PHASE      = 3
N_AMP_PHASE  = 2

history = {'label': [], 'peak': [], 'rms': [], 'snr': []}

def log(label):
    p = s.imager.peak()
    history['label'].append(label)
    history['peak'].append(p['flux'])
    history['rms'].append(p['rms'])
    history['snr'].append(p['snr'])
    print(f"  [{label:22s}]  Pic={p['flux']:.4f}  RMS={p['rms']:.4f}  SNR={p['snr']:.0f}")

print('=== CLEAN + selfcal ===')

s.imager.mapsize(MAP_SIZE, CELLSIZE)
s.imager.invert()
log('Dirty (départ)')

s.imager.peakwin(size=WIN_SIZE)
s.imager.clean(NITER, GAIN)
s.imager.mapsize(MAP_SIZE, CELLSIZE)
s.imager.invert()
log('1er CLEAN')

for i in range(N_PHASE):
    s.imager.selfcal(doamp=False)
    s.imager.mapsize(MAP_SIZE, CELLSIZE)
    s.imager.invert()
    s.imager.delwin()
    s.imager.peakwin(size=WIN_SIZE)
    s.imager.clean(NITER, GAIN)
    s.imager.mapsize(MAP_SIZE, CELLSIZE)
    s.imager.invert()
    log(f'Phase selfcal #{i+1}')

for i in range(N_AMP_PHASE):
    s.imager.selfcal(doamp=True)
    s.imager.mapsize(MAP_SIZE, CELLSIZE)
    s.imager.invert()
    s.imager.delwin()
    s.imager.peakwin(size=WIN_SIZE)
    s.imager.clean(NITER * 2, GAIN)
    s.imager.mapsize(MAP_SIZE, CELLSIZE)
    s.imager.invert()
    log(f'Amp+Phase #{i+1}')

s.imager.restore()
clean_pkg = s.imager.get_map_package(cellsize=CELLSIZE)
p_clean   = s.imager.peak()
log('Clean Map finale')

snr_gain = p_clean['snr'] / history['snr'][0]
print(f'\nGain SNR : ×{snr_gain:.1f}')

### 7 · Convergence

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
x = range(len(history['label']))

ax1.plot(x, history['peak'], 'o-', color='tomato',    lw=1.8, label='Pic (Jy/beam)')
ax1.plot(x, history['rms'],  's--', color='steelblue', lw=1.4, label='RMS (Jy/beam)')
ax1.set_xticks(list(x))
ax1.set_xticklabels(history['label'], rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('Jy/beam')
ax1.set_title('Pic et bruit')
ax1.legend()
ax1.grid(linestyle=':', alpha=0.5)

ax2.plot(x, history['snr'], 'D-', color='seagreen', lw=1.8)
ax2.fill_between(x, history['snr'], alpha=0.15, color='seagreen')
ax2.set_xticks(list(x))
ax2.set_xticklabels(history['label'], rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('SNR')
ax2.set_title('Rapport signal/bruit')
ax2.grid(linestyle=':', alpha=0.5)

plt.suptitle('Convergence CLEAN + selfcal', fontsize=14)
plt.tight_layout()
plt.show()

### 8 · Dirty Map vs Clean Map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pkg, title in [
    (axes[0], dirty,      'Dirty Map'),
    (axes[1], clean_pkg,  'Clean Map (restaurée)'),
]:
    vmax = np.percentile(pkg['data'], 99.8)
    im = ax.imshow(pkg['data'], origin='lower', cmap='afmhot',
                   extent=pkg['extent'], vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('RA offset (mas)')
    ax.set_ylabel('Dec offset (mas)')
    plt.colorbar(im, ax=ax, label='Jy/beam')

plt.suptitle(f"{s.obs.source} — {pol}  |  {MAP_SIZE}×{MAP_SIZE} px, {CELLSIZE} mas/px", fontsize=13)
plt.tight_layout()
plt.show()

### 9 · Sauvegarde et résumé

In [ ]:
output = f"{s.obs.source}_clean.fits"
s.imager.wmap(output)
print(f'Sauvegardé : {output}')

print()
print('=== Résumé ===')
print(f'Source     : {s.obs.source}')
print(f'Pol.       : {pol}')
print(f'Grille     : {MAP_SIZE}×{MAP_SIZE} px  |  {CELLSIZE} mas/px')
info = clean_pkg['info']
print(f'Beam       : {info["bmaj"]:.3f} × {info["bmin"]:.3f} mas  BPA={info["bpa"]:.1f}°')
print(f'Pic final  : {p_clean["flux"]:.4f} Jy/beam')
print(f'RMS        : {info["rms"]:.4f} Jy/beam')
print(f'SNR final  : {p_clean["snr"]:.0f}')

s.cleanup()